In [ ]:
import os
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm

from torchvision.models import (
    resnet18, ResNet18_Weights,
    mobilenet_v3_small, MobileNet_V3_Small_Weights,
)

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
DATA_DIR = "dataset"  

BATCH_SIZE  = 32
NUM_WORKERS = 2  
RANDOM_SEED = 42

In [ ]:
def collect_samples(root):
    class_names = sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)))
    class_to_idx = {cls: i for i, cls in enumerate(class_names)}

    paths, labels = [], []
    for cls in class_names:
        for fname in os.listdir(os.path.join(root, cls)):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                paths.append(os.path.join(root, cls, fname))
                labels.append(class_to_idx[cls])

    return paths, labels, class_names, class_to_idx

In [ ]:
paths, labels, CLASS_NAMES, CLASS_TO_IDX = collect_samples(DATA_DIR)
NUM_CLASSES = len(CLASS_NAMES)

print(f"Clases encontradas: {CLASS_NAMES}")
print(f"Total de imágenes: {len(paths)}")
print()
for cls, idx in CLASS_TO_IDX.items():
    n = labels.count(idx)
    print(f"  [{idx}] {cls}: {n} imágenes")

In [ ]:
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    paths, labels,
    test_size=0.20,
    stratify=labels,
    random_state=RANDOM_SEED,
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels,
    test_size=0.50,
    stratify=temp_labels,
    random_state=RANDOM_SEED,
)

print(f"Train: {len(train_paths)} imagenes")
print(f"Val: {len(val_paths)} imagenes")
print(f"Test: {len(test_paths)} imagenes")

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Train transform:")
print(train_transform)
print("\n")
print("Eval transform:")
print(eval_transform)

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, paths: list, labels: list, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]


train_dataset = CustomImageDataset(train_paths, train_labels, transform=train_transform)
val_dataset = CustomImageDataset(val_paths,   val_labels,   transform=eval_transform)
test_dataset = CustomImageDataset(test_paths,  test_labels,  transform=eval_transform)

print(f"Tamaño train: {len(train_dataset)}")
print(f"Tamaño val: {len(val_dataset)}")
print(f"Tamaño test: {len(test_dataset)}")

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

print(f"Batches en train: {len(train_loader)}")
print(f"Batches en val: {len(val_loader)}")
print(f"Batches en test: {len(test_loader)}")

In [ ]:
def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    t = tensor.clone()
    for c, (m, s) in enumerate(zip(mean, std)):
        t[c] = t[c] * s + m
    return t.clamp(0, 1)


imgs, lbls = next(iter(train_loader))
n_show = min(16, len(imgs))

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    if i >= n_show:
        ax.axis("off")
        continue
    img = denormalize(imgs[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[lbls[i].item()], fontsize=9)
    ax.axis("off")

plt.suptitle("Primer batch de train (desnormalizado)", y=1.01)
plt.tight_layout()
plt.show()

print(f"\nShape del batch de imágenes : {imgs.shape}")
print(f"Shape del batch de etiquetas: {lbls.shape}")